In [0]:
# Check the cluster
# v2 
attached_cluster_name = spark.conf.get(
    "spark.databricks.clusterUsageTags.clusterName", ""
)
if not attached_cluster_name.endswith("uc_support") and not (
    attached_cluster_name.startswith("bfdw_")
    and "compute_uc_jobs" in attached_cluster_name
):
    raise Exception(
        "This notebook is being executed in an incorrect cluster. Please attach it to the *uc_support cluster or one of the bfdw_*compute_uc_jobs* clusters"
    )
else:
    print(f"Cluster is: {attached_cluster_name}")

2.Code to select catolog based on the workspace

In [0]:
workspace_catalogs = [
    e.catalog.lower()
    for e in spark.sql(f"SHOW CATALOGS").collect()
    if e.catalog not in ["main", "samples", "system", "__databricks_internal"]
]
print(f"Catalogs in the workspace: {workspace_catalogs}")
banfield_catalogs = ["banfield_catalogdev", "banfield_catalogtst", "banfield_catalog"]
target_catalog0 = [e for e in banfield_catalogs if e in workspace_catalogs]
if (not target_catalog0) or len(target_catalog0) != 1:
    raise Exception(
        f"Expecting any one of the active banfield catalog but received {len(target_catalog0)}; Banfield catalog names: {banfield_catalogs}"
    )
spark.conf.set("catlg.banfield_catalog", target_catalog0[0])
print(f"catlg.banfield_catalog: {target_catalog0[0]}")
 
bf_vwmvhcores = ["bf_vwmvhcoredev", "bf_vwmvhcoretst", "bf_vwmvhcore"]
target_catalog1 = [e for e in bf_vwmvhcores if e in workspace_catalogs]
if (not target_catalog1) or len(target_catalog1) != 1:
    raise Exception(
        f"Expecting any one of the active bf_vwmvhcore but received {len(target_catalog1)}; bf_vwmvhcore names: {bf_vwmvhcores}"
    )
spark.conf.set("catlg.bf_vwmvhcore", target_catalog1[0])
print(f"catlg.bf_vwmvhcore: {target_catalog1[0]}")


bf_vwedhs = ["bf_vwedhdev", "bf_vwedhtst", "bf_vwedh"]
target_catalog2 = [e for e in bf_vwedhs if e in workspace_catalogs]
if (not target_catalog1) or len(target_catalog2) != 1:
    raise Exception(
        f"Expecting any one of the active bf_vwmvhcore but received {len(target_catalog2)}; bf_vwmvhcore names: {bf_vwedhs}"
    )
spark.conf.set("catlg.bf_vwedh", target_catalog2[0])
print(f"catlg.bf_vwedh: {target_catalog2[0]}")

bf_vwvoyagers = ["bf_vwvoyagerdev", "bf_vwvoyagertst", "bf_vwvoyager"]
target_catalog3 = [e for e in bf_vwvoyagers if e in workspace_catalogs]
if (not target_catalog1) or len(target_catalog3) != 1:
    raise Exception(
        f"Expecting any one of the active bf_vwmvhcore but received {len(target_catalog3)}; bf_vwvoyager names: {bf_vwvoyagers}"
    )
spark.conf.set("catlg.bf_vwvoyager", target_catalog3[0])
print(f"catlg.bf_vwvoyager: {target_catalog3[0]}")

3. Create table Bronze Layer

In [0]:
%sql
create or replace table ${catlg.banfield_catalog}.bfdw_bronze.cmn_tbcmdcurrentitem (
inventoryid			bigint comment 'name: inventory identifier
description: unique identifier of an item in petware.
source: petware pmri
dwpetnet.inventory.inventoryid', 
inventoryrootid		bigint comment 'name: inventory root identifier
description: the non regionalized parent of regionalized inventoryids (i.e. all regionalized inventory, each having a unique inventoryid, will have a common inventoryrootid).
source: petware pmri
dwpetnet.inventory.inventoryrootid', 
item_descr			string comment 'name: item description
description: the textual us english description of the item.
source: petware pmri
dwpetnet.inventory.description', 
item_class			string comment 'name: item class
description: a high-level item classification.
bundle
complex
item
subprot
source: petware pmri
dwpetnet.inventory.itemclass', 
item_primary_typ	string comment 'name: item primary type
description: primary inventory type.
1             	services
12            	non-service items
2             	diagnostics
3             	medications
6             	surgical procedures
8             	medical/health care products
9             	wellness plans

source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventorytype.inventoryid
dwpetnet.inventorytype.description', 
item_secondary_typ	string comment 'name: item secondary type
description: secondary inventory type.
ex:
32	dental care product
35	heartworm medication
36	intestinal medication
37	medical supply

source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventorycategory.inventoryid
dwpetnet.inventorycategory.description', 
item_status			string comment 'name: item status
description: item has active status only if the given date is between inventory area record start date and inventory area record end date and also between inventory start date and inventory end date. else, inactive.
only active items are available for the hospitals to order.
active or inactive
source: petware pmri
dwpetnet.inventory.recordstartdate
dwpetnet.inventory.recordenddate
dwpetnet.inventory.startdate
dwpetnet.inventory.enddate', 
prdct_or_srvc		string comment 'name: product or service
description: designation of the item as a product or a service.
other
serv
prod

source: petware pmri
dwpetnet.inventory.productorservice', 
drug_desc			string comment 'name: drug description
description: the description of the prescription drug.
source: petware pmri
dwpetnet.drug.description
uses the following join conditions
dwpetnet.inventory.inventoryrootid = dwpetnet.inventorydrug.inventoryrootid 
dwpetnet.inventorydrug.medicationid = dwpetnet.medication.medicationid
dwpetnet.medication.drugformid = dwpetnet.drugform.drugformid
dwpetnet.drugform.drugid  = dwpetnet.drug.drugid', 
item_coupon_typ		string comment 'name: item coupon type
description: the type for coupon items.
banfield coupon - physical
banfield coupon - virtual (ie. yellow page)
not a coupon
numbered banfield coupon
numbered coupon
physical coupon
serialized banfield coupon
serialized coupon
virtual coupon
source: petware pmri
dwpetnet.inventory.coupontype
dwpetnet.inventory.coupontypeid
dwpetnet.coupontype.description', 

price_capped_bndl_mbr_flg string not null comment 'name: price capped bundle member flag
description: a flag to indicate whether or not the item is a member of a price-capped bundle
y = yes, n = no
source: petware pmri
dwpetnet.inventory.ispricecap', 
default_unit_price_lvl_01 bigint comment 'name: default unit price level 1
description: item unit price (level 1 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_unit_price_lvl_02 bigint comment 'name: default unit price level 2
description: item unit price (level 2 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_unit_price_lvl_03 bigint comment 'name: default unit price level 3
description: item unit price (level 3 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_unit_price_lvl_04 bigint comment 'name: default unit price level 4
description: item unit price (level 4 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_unit_price_lvl_05 bigint comment 'name: default unit price level 5
description: item unit price (level 5 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_unit_price_lvl_06 bigint comment 'name: default unit price level 6
description: item unit price (level 6 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_unit_price_lvl_07 bigint comment 'name: default unit price level 7
description: item unit price (level 7 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_unit_price_lvl_08 bigint comment 'name: default unit price level 8
description: item unit price (level 8 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_unit_price_lvl_09 bigint comment 'name: default unit price level 9
description: item unit price (level 9 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
item_taxable_flg string not null comment 'name: item taxable flag
description: a flag to indicate whether or not the item is taxable
y = yes, n = no
source: petware pmri
dwpetnet.inventory.issalestaxable', 
pricing_rule string comment 'name: pricing rule
description: the rule used to determine pricing structure for the item.
ex:
coupon
flat
graduate
na_disc
na_zero
negative - linked product
negative - linked service

source: petware pmri
dwpetnet.pricerule.description
dwpetnet.pricerule.inventoryid
dwpetnet.inventory.inventoryid', 
license_tag_fee_flg string not null comment 'name: license tag fee flag
description: flag to indicate whether or not the item represents a fee for a license tag.
y = yes, n = no
source: petware pmri
dwpetnet.inventory.islicenseitem', 
item_prev_care_flg string not null comment 'name: item preventive care flag
description: a flag to indicate whether or not medical quality assurance considers this item as a preventive care item.  records with a value of "y" will be included in the refresh of the pet visit current preventive care group entity.
source: petware pmri
dwpetnet.inventory.inventoryrootid
dwpetnet.prevcareentityinventorylink.inventoryrootid
dwpetnet.prevcareentityinventorylink.defaultusefuldays', 
dea_controlled_flg string not null comment 'name: drug enforcement administration
description: a flag to indicate whether or not the item is controlled by the drug enforcement agency.
y = yes, n = no
source: petware pmri
dwpetnet.inventory.isdeacontrolled', 
item_male_flg string not null comment 'name: item male flag
description: a flag to indicate whether or not the item is suitable for male pets.
y = yes, n = no
source: petware pmri
dwpetnet.inventory.isformale', 
item_female_flg string not null comment 'name: item female flag
description: a flag to indicate whether or not the item is suitable for female pets.
y = yes, n = no
source: petware pmri
dwpetnet.inventory.isforfemale', 
item_neuter_flg string not null comment 'name: item neuter flag
description: a flag to indicate whether or not the item is suitable for neutered pets.
y = yes, n = no
source: petware pmri
dwpetnet.inventory.isforneutered', 
item_spay_flg string not null comment 'name: item spay flag
description: a flag to indicate whether or not the item is suitable for spayed pets.
y = yes, n = no
source: petware pmri
dwpetnet.inventory.isforspayed', 
item_canine_flg string not null comment 'name: item canine flag
description: a flag to indicate whether or not the item is suitable for canines.
y = yes, n = no
source: petware pmri
dwpetnet.inventory.islicenseitem', 
item_feline_flg string not null comment 'name: item feline flag
description: a flag to indicate whether or not the item is suitable for male felines.
y = yes, n = no
source: petware pmri
dwpetnet.inventory.islicenseitem', 
item_primary_typ_cd string comment 'name: item primary type code
description: system short name for the primary inventory type.
1             	services
12            	non-service items
2             	diagnostics
3             	medications
6             	surgical procedures
8             	medical/health care products
9             	wellness plans

source: petware pmri
dwpetnet.inventory.inventorytypeid', 
item_secondary_typ_cd string comment 'name: item secondary type code
description: system short name for the secondary inventory type.
ex:
32	dental care product
35	heartworm medication
36	intestinal medication
37	medical supply

source: petware pmri
dwpetnet.inventory.inventorycategoryid', 
dw_deleted_ind bigint not null comment 'name: data warehouse deleted indicator
description: dw metadata. an indicator set to 1 to designate when a particular key has been removed from a source system of record.
a value of 0 indicates that the key is still present in the source system of record.
source: derived by dw load process, by comparing source keys against target keys.', 
dw_job_id bigint not null comment 'name: data warehouse job identifier
description: dw metadata. an identifier for the data warehouse load process that last affected the record.
source: data warehouse database sequence generator', 
dw_load_dt timestamp not null comment 'name: data warehouse load date
description: dw metadata. the date the data warehouse load process that last affected the record.
this is the starting date/time of the process that last inserted or updated the record.
source: data warehouse local system date/time (us-pacific time zone).', 
audit_category_cd string comment 'name: audit category code
description: item audit category code.
a      	reg. adjust
a      	adjustment
j      	wp adjust
j      	wp adjustment (obsolete)
m      	wp member
m      	wp membership fee
n      	none
n      	normal
p      	wp payment (monthlyear)
p      	wp payment
r      	refund request
t      	sales tax
source: petware pmri
dwpetnet.inventory.auditcategorycode', 
audit_category_desc string comment 'name: audit category description
description: item audit category description.
a      	reg. adjust
a      	adjustment
j      	wp adjust
j      	wp adjustment (obsolete)
m      	wp member
m      	wp membership fee
n      	none
n      	normal
p      	wp payment (monthlyear)
p      	wp payment
r      	refund request
t      	sales tax
source: petware pmri
dwpetnet.auditcategorycode.description
dwpetnet.auditcategorycode.auditcategorycode
dwpetnet.inventory.auditcategorycode', 
default_packing_price_lvl_01 bigint comment 'name: default packing price level 1
description: item package price (level 1 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_packing_price_lvl_02 bigint comment 'name: default packing price level 2
description: item package price (level 2 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_packing_price_lvl_03 bigint comment 'name: default packing price level 3
description: item package price (level 3 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_packing_price_lvl_04 bigint comment 'name: default packing price level 4
description: item package price (level 4 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_packing_price_lvl_05 bigint comment 'name: default packing price level 5
description: item package price (level 5 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_packing_price_lvl_06 bigint comment 'name: default packing price level 6
description: item package price (level 6 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_packing_price_lvl_07 bigint comment 'name: default packing price level 7
description: item package price (level 7 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
reference_lab_flg string not null comment 'name: reference lab flag
description: a flag to indicate whether or not the item is a reference lab diagnostics test that will be sent out to an external lab.
y = yes, n = no
source: petware pmri
dwpetnet.inventory.isreferencelabtest', 
default_packing_price_lvl_08 bigint comment 'name: default packing price level 8
description: item package price (level 8 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_packing_price_lvl_09 bigint comment 'name: default packing price level 9
description: item package price (level 9 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_packing_price_lvl_10 bigint comment 'name: default packing price level 10
description: item package price (level 10 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_10 bigint comment 'name: default unit price level 10
description: item unit price (level 10 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_11 bigint comment 'name: default packing price level 11
description: item package price (level 11 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_11 bigint comment 'name: default unit price level 11
description: item unit price (level 11 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_12 bigint comment 'name: default packing price level 12
description: item package price (level 12 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_12 bigint comment 'name: default unit price level 12
description: item unit price (level 12 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_13 bigint comment 'name: default packing price level 13
description: item package price (level 13 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_13 bigint comment 'name: default unit price level 13
description: item unit price (level 13 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_14 bigint comment 'name: default packing price level 14
description: item package price (level 14 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_14 bigint comment 'name: default unit price level 14
description: item unit price (level 14 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_15 bigint comment 'name: default packing price level 15
description: item package price (level 15 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_15 bigint comment 'name: default unit price level 15
description: item unit price (level 15 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_16 bigint comment 'name: default packing price level 16
description: item package price (level 16 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_16 bigint comment 'name: default unit price level 16
description: item unit price (level 16 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_17 bigint comment 'name: default packing price level 17
description: item package price (level 17 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_17 bigint comment 'name: default unit price level 17
description: item unit price (level 17 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_18 bigint comment 'name: default packing price level 18
description: item package price (level 18 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_18 bigint comment 'name: default unit price level 18
description: item unit price (level 18 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_19 bigint comment 'name: default packing price level 19
description: item package price (level 19 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_19 bigint comment 'name: default unit price level 19
description: item unit price (level 19 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_20 bigint comment 'name: default packing price level 20
description: item package price (level 20 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_20 bigint comment 'name: default unit price level 20
description: item unit price (level 20 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
prdct_cost_amt bigint comment 'name: product cost amount
description: the product cost amount of the item.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventorycost.inventoryid
dwpetnet.inventorycost.productcost', 
default_packing_price_lvl_21 bigint comment 'name: default packing price level 21
description: item package price (level 21 us) - home delivery for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_21 bigint comment 'name: default unit price level 21
description: item unit price (level 21 us) - home delivery for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_22 bigint comment 'name: default packing price level 22
description: item package price (level 22 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_22 bigint comment 'name: default unit price level 22
description: item unit price (level 22 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_23 bigint comment 'name: default packing price level 23
description: item package price (level 23 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_23 bigint comment 'name: default unit price level 23
description: item unit price (level 23 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_24 bigint comment 'name: default packing price level 24
description: item package price (level 24 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_24 bigint comment 'name: default unit price level 24
description: item unit price (level 24 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_25 bigint comment 'name: default packing price level 25
description: item package price (level 25 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_25 bigint comment 'name: default unit price level 25
description: item unit price (level 25 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 

item_gl_account_num string comment 'name: item general ledger account number
description: the gl account number associated with the item
source: petware pmri
dwmisc.mmi_hr_gl_account.gl_account_num
dwmisc.mmi_hr_gl_account.pmri_inventory_category
dwpetnet.inventorycategory.description
dwpetnet.inventorycategory.inventorycategoryid
dwpetnet.inventory.inventorycategoryid', 
item_gl_account_descr string comment 'name: item general ledger account description
description: description of the gl account associated with the item
source: petware pmri
dwmisc.mmi_hr_gl_account.gl_account_descr
dwmisc.mmi_hr_gl_account.pmri_inventory_category
dwpetnet.inventorycategory.description
dwpetnet.inventorycategory.inventorycategoryid
dwpetnet.inventory.inventorycategoryid', 
item_updt_cost_with_price_flg string comment 'name: item update cost with price flag
description: indicates if the cost of goods field should be updated with the user entered unitprice value or if it should remain at the pmri set price
y = yes, n = no
source: petware pmri
dwpetnet.inventory.isupdatecostwithprice', 
item_user_editable_flg string comment 'name: item user editable flag
description: indicates if the users can control the unitprice assigned to a particular item via petware
y = yes, n = no
source: petware pmri
dwpetnet.inventory.isusereitable', 
non_sx_anesthesia_flg string comment 'name: non surgery anesthesia flag
description: indicates if anesthesia was used for a non-surgical procedure (e.g. y or n)
source: petware pmri
dwmisc.item_sx_anesthesia_excluded.non_sx_anesthesia_flg
dwmisc.item_sx_anesthesia_excluded.inventoryid
dwpetnet.inventory.inventoryid', 
healthy_sx_bndl_flg string comment 'name: healthy sex bundle flag
description: indicates if the procedure was done on a healthy pet; basically an elective procedure (e.g. y or n)
source: petware pmri
dwmisc.item_healthy_sx_bundle.healthy_sx_bndl_flg
dwmisc.item_healthy_sx_bundle.inventoryid
dwpetnet.inventory.inventoryid', 
item_tertiary_typ_cd bigint comment 'name: item tertiary type code
description: numeric value that corresponds to the item_tertiary_type (e.g. 2 is abdominal, 88 is dental prophylaxis)
ex:
44	antiviral
45	anxiolytic
46	appetite stimulant
48	behavior
49	benzodiazepine

source: petware pmri
dwpetnet.inventory.inventorytertiarycategoryid', 
item_tertiary_typ string comment 'name: item tertiary type
description: a grouping for medicine to use in reviewing items (e.g. abdominal, dental prophylaxis etc)
ex:
44	antiviral
45	anxiolytic
46	appetite stimulant
48	behavior
49	benzodiazepine

source: petware pmri
dwpetnet.inventory.inventorytertiarycategoryid
dwpetnet.inventorytertiarycategory.inventorytertiarycategoryid
dwpetnet.inventorytertiarycategory.description', 
management_category_cd bigint comment 'name: management category code
description: numeric value that corresponds to the management_category
ex:
1	anesthesia
2	behavior
3	cardiopulmonary
4	dental
5	dermatology
6	diagnostics
source: petware pmri
dwpetnet.managementcategory.managementcategoryid
dwpetnet.inventorymanagementcategorylink.managementcategoryid
dwpetnet.inventorymanagementcategorylink.inventoryrootid
dwpetnet.inventory.inventoryrootid', 
management_category string comment 'name: management category
description: a grouping for category management to use in reviewing items
ex:
1	anesthesia
2	behavior
3	cardiopulmonary
4	dental
5	dermatology
6	diagnostics
source: petware pmri
dwpetnet.managementcategory.description
dwpetnet.managementcategory.managementcategoryid
dwpetnet.inventorymanagementcategorylink.managementcategoryid
dwpetnet.inventorymanagementcategorylink.inventoryrootid
dwpetnet.inventory.inventoryrootid', 
default_mfg_nam string comment 'name: default manufacturer name
description: the default vendor name for same inventoryvendorlinkareaid as in inventory areaid, otherwise the default vendor name for global areaid else null
source: petware pmri
dwpetnet.vendor.name
dwpetnet.vendor.vendorid
dwpetnet.inventoryvendorlink.vendorid
dwpetnet.inventoryvendorlink.inventoryrootid
dwpetnet.inventoryvendorlink.areaid
dwpetnet.inventory.inventoryrootid
dwpetnet.inventory.areaid', 
documentation_cd bigint comment 'name: documentation code
description: numeric value corresponding to documentation_cd_desc which indicates if the item requires a label and/or a medical note (value, 1,2,3,4)
source: petware pmri
dwpetnet.inventory.documentationcode', 
documentation_cd_desc string comment 'name: documentation code description
description: indicates if the item requires a label and/or a medical note (values: nolabel_nomedicalnote, requiredlabel_requiredmedicalnote, optionallabel_requiredmedicalnote, and nolabel_requiredmedicalnote)
source: petware pmri
dwpetnet.codedescription.standarddescription
dwpetnet.codedescription.numericcode
dwpetnet.inventory.documentationcode', 
fda_prescription_flg string comment 'name: food and drug administration prescription flag
description: a flag (y or n) to indicate whether or not the item is an fda prescription
source: petware pmri
dwpetnet.inventory.isfdaprescription', 
item_formulary_nam string comment 'name: item formulary name
description: name of the method of delivery for medicines (e.g. bottle, capsule, injection, premeasured vial etc).
source: petware pmri
dwpetnet.unit.description
uses the following join conditions
dwpetnet.inventory.inventoryrootid = dwpetnet.inventorydrug.inventoryrootid
dwpetnet.inventorydrug.doseunitid = dwpetnet.unit.unitid', 
product_grp_nam string comment 'name: product group name
description: the group name of certain items (e.g. advantage, flea products, proheart etc). many values are null.
source: petware pmri
dwpetnet.rulecategory.description
uses the following join conditions
dwpetnet.inventory.inventoryrootid = dwpetnet.inventoryrulecategorylink.inventoryrootid
dwpetnet.inventoryrulecategorylink.rulecategoryid = dwpetnet.rulecategory.rulecategoryid', 
prev_care_useful_life_days bigint comment 'name: preventive care useful life days
description: the default value for the useful days of life for a given item.
source: petware pmri
dwpetnet.prevcareentityinventorylink.defaultusefuldays', 
item_unit string comment 'name: item unit
description: the description of the unit format for the item
ex:
bag
bottle
box
can
card
case

source: petware pmri
dwpetnet.inventory.note', 
item_src_system_cd string not null comment 'name: item source system code
description: an abbreviation to identify the source system where the record was found.  
valid values include:
   dw - data warehouse derived record;
   pn - petware (.net records);
   pw - petware (older foxpro records)', 
multiple_price_area_flg string comment 'name: multiple price area flag
description: a flag that is set to y if there are multiple price areas for a given inventoryid.
source: petware pmri
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventory.inventoryid', 
ft_hw_item_dose_cnt bigint comment 'name: flea tick heartworm item dose count
description: valid only for the 3 item secondary types: flea/tick medication, heartworm medication and flea/tick/heartworm.
source: petware pmri
trunc(max(dwpetnet.prevcareentityinventorylink.defaultusefuldays) / 30) ', 
delivery_method_nam string comment 'name: delivery method name
description: deliverymethod name. possible to get multiple delivery methods for given item.
source: petware pmri
dwpetnet.inventorydeliverymethodlink.deliverymethodid
dwpetnet.deliverymethod.deliverymethodid
dwpetnetdeliverymethod.description', 
dw_item_descriptiontext_id bigint comment 'name: data warehouse item description text identifier
description: textid that corresponds to a text used in petware.
source: petware pmri
inventory.descriptiontextid', 
item_descr_spanish_us string comment 'name: item description spanish united states
description: the textual us spanish description of the item.
source: petware pmri
inventory.descriptiontextid
dwpetnet.translatedtext.textid
dwpetnet.translatedtext.cultureid
cmd.tb_cmd_translated_text.spanish_us_text', 
item_descr_spanish_pr string comment 'name: item description spanish puerto rico
description: the textual puertorico spanish description of the item.
source: petware pmri
inventory.descriptiontextid
dwpetnet.translatedtext.textid
dwpetnet.translatedtext.cultureid
cmd.tb_cmd_translated_text.spanish_pr_text', 
item_areaid bigint comment 'name: item area identifier
description: areaid that belongs to the inventoryid.
source: petware pmri
dwpetnet.inventory.areaid', 
item_area_record_start_dt timestamp comment 'name: item area record start date
description: inventory area record start date.
source: petware pmri
dwpetnet.inventory.recordstartdate', 
item_area_record_end_dt timestamp comment 'name: item area record end date
description: inventory area record end date.
source: petware pmri
dwpetnet.inventory.reordenddate', 
item_start_dt timestamp comment 'name: item start date
description: inventoryid start date. when the item comes into effective
source: petware pmri
dwpetnet.inventory.startdate', 
item_end_dt timestamp comment 'name: item end date
description: inventoryid end date. when the item has expired and is no longer in use
source: petware pmri
dwpetnet.inventory.enddate', 
drug_form_desc string comment 'name: drug form description
description: the  "form" of the prescription drug in which the medication was provided in, be it topical, oral, etc. for diet it would be like cardiac.
source: petware pmri
dwpetnet.drugformdescription.description
uses the following join conditions
dwpetnet.inventory.inventoryrootid = dwpetnet.inventorydrug.inventoryrootid 
dwpetnet.inventorydrug.medicationid = dwpetnet.medication.medicationid
dwpetnet.medication.drugformid = dwpetnet.drugform.drugformid
dwpetnet.drugform.drugformdescriptionid  = dwpetnet.drugformdescription.drugformdescriptionid', 
default_packing_price_lvl_26 bigint comment 'name: default packing price level 26
description: item package price (level 26 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_26 bigint comment 'name: default unit price level 26
description: item unit price (level 26 us) - home delivery for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_27 bigint comment 'name: default packing price level 27
description: item package price (level 27 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_27 bigint comment 'name: default unit price level 27
description: item unit price (level 27 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_28 bigint comment 'name: default packing price level 28
description: item package price (level 28 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_28 bigint comment 'name: default unit price level 28
description: item unit price (level 28 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_29 bigint comment 'name: default packing price level 29
description: item package price (level 29 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_29 bigint comment 'name: default unit price level 29
description: item unit price (level 29 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_30 bigint comment 'name: default packing price level 30
description: item package price (level 30 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_30 bigint comment 'name: default unit price level 30
description: item unit price (level 30 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_31 bigint comment 'name: default packing price level 31
description: item package price (level 31 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_31 bigint comment 'name: default unit price level 31
description: item unit price (level 31 us) - home delivery for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_32 bigint comment 'name: default packing price level 32
description: item package price (level 32 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_32 bigint comment 'name: default unit price level 32
description: item unit price (level 32 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_33 bigint comment 'name: default packing price level 33
description: item package price (level 33 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_33 bigint comment 'name: default unit price level 33
description: item unit price (level 33 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_34 bigint comment 'name: default packing price level 34
description: item package price (level 34 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_34 bigint comment 'name: default unit price level 34
description: item unit price (level 34 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_35 bigint comment 'name: default packing price level 35
description: item package price (level 35 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_35 bigint comment 'name: default unit price level 35
description: item unit price (level 35 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_36 bigint comment 'name: default packing price level 36
description: item package price (level 36 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_36 bigint comment 'name: default unit price level 36
description: item unit price (level 36 us) - home delivery for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_37 bigint comment 'name: default packing price level 37
description: item package price (level 37 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_37 bigint comment 'name: default unit price level 37
description: item unit price (level 37 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_38 bigint comment 'name: default packing price level 38
description: item package price (level 38 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_38 bigint comment 'name: default unit price level 38
description: item unit price (level 38 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_39 bigint comment 'name: default packing price level 39
description: item package price (level 39 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_39 bigint comment 'name: default unit price level 39
description: item unit price (level 39 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_40 bigint comment 'name: default packing price level 40
description: item package price (level 40 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_40 bigint comment 'name: default unit price level 40
description: item unit price (level 40 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_41 bigint comment 'name: default packing price level 41
description: item package price (level 41 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_41 bigint comment 'name: default unit price level 41
description: item unit price (level 41 us) - home delivery for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_42 bigint comment 'name: default packing price level 42
description: item package price (level 42 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_42 bigint comment 'name: default unit price level 42
description: item unit price (level 42 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_43 bigint comment 'name: default packing price level 43
description: item package price (level 43 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_43 bigint comment 'name: default unit price level 43
description: item unit price (level 43 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_44 bigint comment 'name: default packing price level 44
description: item package price (level 44 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_44 bigint comment 'name: default unit price level 44
description: item unit price (level 44 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_45 bigint comment 'name: default packing price level 45
description: item package price (level 45 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_45 bigint comment 'name: default unit price level 45
description: item unit price (level 45 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_46 bigint comment 'name: default packing price level 46
description: item package price (level 46 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_46 bigint comment 'name: default unit price level 46
description: item unit price (level 46 us) - home delivery for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_47 bigint comment 'name: default packing price level 47
description: item package price (level 47 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_47 bigint comment 'name: default unit price level 47
description: item unit price (level 47 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_48 bigint comment 'name: default packing price level 48
description: item package price (level 48 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_48 bigint comment 'name: default unit price level 48
description: item unit price (level 48 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_49 bigint comment 'name: default packing price level 49
description: item package price (level 49 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_49 bigint comment 'name: default unit price level 49
description: item unit price (level 49 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_50 bigint comment 'name: default packing price level 50
description: item package price (level 50 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_50 bigint comment 'name: default unit price level 50
description: item unit price (level 50 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_51 bigint comment 'name: default packing price level 51
description: item package price (level 51 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_51 bigint comment 'name: default unit price level 51
description: item unit price (level 51 us) - home delivery for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_52 bigint comment 'name: default packing price level 52
description: item package price (level 52 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_52 bigint comment 'name: default unit price level 52
description: item unit price (level 52 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_53 bigint comment 'name: default packing price level 53
description: item package price (level 53 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_53 bigint comment 'name: default unit price level 53
description: item unit price (level 53 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_54 bigint comment 'name: default packing price level 54
description: item package price (level 54 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_54 bigint comment 'name: default unit price level 54
description: item unit price (level 54 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_55 bigint comment 'name: default packing price level 55
description: item package price (level 55 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_55 bigint comment 'name: default unit price level 55
description: item unit price (level 55 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_56 bigint comment 'name: default packing price level 56
description: item package price (level 56 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_56 bigint comment 'name: default unit price level 56
description: item unit price (level 56 us) - home delivery for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_57 bigint comment 'name: default packing price level 57
description: item package price (level 57 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_57 bigint comment 'name: default unit price level 57
description: item unit price (level 57 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_58 bigint comment 'name: default packing price level 58
description: item package price (level 58 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_58 bigint comment 'name: default unit price level 58
description: item unit price (level 58 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_59 bigint comment 'name: default packing price level 59
description: item package price (level 59 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_59 bigint comment 'name: default unit price level 59
description: item unit price (level 59 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_60 bigint comment 'name: default packing price level 60
description: item package price (level 60 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_60 bigint comment 'name: default unit price level 60
description: item unit price (level 60 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_61 bigint comment 'name: default packing price level 61
description: item package price (level 61 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_61 bigint comment 'name: default unit price level 61
description: item unit price (level 61 us) - home delivery for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_62 bigint comment 'name: default packing price level 62
description: item package price (level 62 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_62 bigint comment 'name: default unit price level 62
description: item unit price (level 62 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_63 bigint comment 'name: default packing price level 63
description: item package price (level 63 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_63 bigint comment 'name: default unit price level 63
description: item unit price (level 63 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_64 bigint comment 'name: default packing price level 64
description: item package price (level 64 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_64 bigint comment 'name: default unit price level 64
description: item unit price (level 64 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_65 bigint comment 'name: default packing price level 65
description: item package price (level 65 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_65 bigint comment 'name: default unit price level 65
description: item unit price (level 65 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
item_shippable_flg string   comment  'name: item shippable flag 
description: a flag to indicate whether or not the item is shippable.
source:
dwpetnet.inventory.isshippable',

fw_createdts timestamp  comment 'the timestamp when the record was first loaded into the iron table',
fw_modifiedts timestamp  comment 'the timestamp when the record was modified and loaded in bronze table',
fw_filename string comment 'the name of the file from which the record was loaded',
rowhash string comment 'hash value generated from the contents of each row, used for detecting changes, deduplication, and maintaining data integrity in Delta Lake tables',
record_status string comment 'reflects the status of the bronze layer record, such as valid or invalid. we will push only valid records to silver layer',
constraint `cmn_tbcmdcurrentitem_pk` primary key (`inventoryid`) rely)
using delta
comment 'name: inventory item dimension
description: the set of distinct items and services that may be referenced as a line item on an invoice.  this includes taxes, coupons and adjustments, as well as products and services.
grain: one row per inventory item, per historical change for that item
each inventory item will have multiple rows in this table. with a row for each change to any of its attributes.
each historical instance of a row will have an effective start and end date
the most recent version of a row is marked with a dw_curr_row_ind = 1
this table is loaded every day. note that currently, pmri is updated only every few weeks or so. in between releases, it is unlikely that any changes will occur in this table.
source: petware pmri
dwpetnet.inventory'
tblproperties (
  'delta.checkpoint.writestatsasjson' = 'false',
  'delta.checkpoint.writestatsasstruct' = 'true',
  'delta.minreaderversion' = '1',
  'delta.minwriterversion' = '2',
  'delta.feature.allowColumnDefaults' = 'supported')

4. Create table for Silver Layer

In [0]:
%sql
create or replace table ${catlg.banfield_catalog}.bfdw_silver.cmn_tbcmdcurrentitem (
inventoryid			bigint comment 'name: inventory identifier
description: unique identifier of an item in petware.
source: petware pmri
dwpetnet.inventory.inventoryid', 
inventoryrootid		bigint comment 'name: inventory root identifier
description: the non regionalized parent of regionalized inventoryids (i.e. all regionalized inventory, each having a unique inventoryid, will have a common inventoryrootid).
source: petware pmri
dwpetnet.inventory.inventoryrootid', 
item_descr			string comment 'name: item description
description: the textual us english description of the item.
source: petware pmri
dwpetnet.inventory.description', 
item_class			string comment 'name: item class
description: a high-level item classification.
bundle
complex
item
subprot
source: petware pmri
dwpetnet.inventory.itemclass', 
item_primary_typ	string comment 'name: item primary type
description: primary inventory type.
1             	services
12            	non-service items
2             	diagnostics
3             	medications
6             	surgical procedures
8             	medical/health care products
9             	wellness plans

source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventorytype.inventoryid
dwpetnet.inventorytype.description', 
item_secondary_typ	string comment 'name: item secondary type
description: secondary inventory type.
ex:
32	dental care product
35	heartworm medication
36	intestinal medication
37	medical supply

source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventorycategory.inventoryid
dwpetnet.inventorycategory.description', 
item_status			string comment 'name: item status
description: item has active status only if the given date is between inventory area record start date and inventory area record end date and also between inventory start date and inventory end date. else, inactive.
only active items are available for the hospitals to order.
active or inactive
source: petware pmri
dwpetnet.inventory.recordstartdate
dwpetnet.inventory.recordenddate
dwpetnet.inventory.startdate
dwpetnet.inventory.enddate', 
prdct_or_srvc		string comment 'name: product or service
description: designation of the item as a product or a service.
other
serv
prod

source: petware pmri
dwpetnet.inventory.productorservice', 
drug_desc			string comment 'name: drug description
description: the description of the prescription drug.
source: petware pmri
dwpetnet.drug.description
uses the following join conditions
dwpetnet.inventory.inventoryrootid = dwpetnet.inventorydrug.inventoryrootid 
dwpetnet.inventorydrug.medicationid = dwpetnet.medication.medicationid
dwpetnet.medication.drugformid = dwpetnet.drugform.drugformid
dwpetnet.drugform.drugid  = dwpetnet.drug.drugid', 
item_coupon_typ		string comment 'name: item coupon type
description: the type for coupon items.
banfield coupon - physical
banfield coupon - virtual (ie. yellow page)
not a coupon
numbered banfield coupon
numbered coupon
physical coupon
serialized banfield coupon
serialized coupon
virtual coupon
source: petware pmri
dwpetnet.inventory.coupontype
dwpetnet.inventory.coupontypeid
dwpetnet.coupontype.description', 

price_capped_bndl_mbr_flg string not null comment 'name: price capped bundle member flag
description: a flag to indicate whether or not the item is a member of a price-capped bundle
y = yes, n = no
source: petware pmri
dwpetnet.inventory.ispricecap', 
default_unit_price_lvl_01 bigint comment 'name: default unit price level 1
description: item unit price (level 1 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_unit_price_lvl_02 bigint comment 'name: default unit price level 2
description: item unit price (level 2 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_unit_price_lvl_03 bigint comment 'name: default unit price level 3
description: item unit price (level 3 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_unit_price_lvl_04 bigint comment 'name: default unit price level 4
description: item unit price (level 4 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_unit_price_lvl_05 bigint comment 'name: default unit price level 5
description: item unit price (level 5 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_unit_price_lvl_06 bigint comment 'name: default unit price level 6
description: item unit price (level 6 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_unit_price_lvl_07 bigint comment 'name: default unit price level 7
description: item unit price (level 7 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_unit_price_lvl_08 bigint comment 'name: default unit price level 8
description: item unit price (level 8 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_unit_price_lvl_09 bigint comment 'name: default unit price level 9
description: item unit price (level 9 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
item_taxable_flg string not null comment 'name: item taxable flag
description: a flag to indicate whether or not the item is taxable
y = yes, n = no
source: petware pmri
dwpetnet.inventory.issalestaxable', 
pricing_rule string comment 'name: pricing rule
description: the rule used to determine pricing structure for the item.
ex:
coupon
flat
graduate
na_disc
na_zero
negative - linked product
negative - linked service

source: petware pmri
dwpetnet.pricerule.description
dwpetnet.pricerule.inventoryid
dwpetnet.inventory.inventoryid', 
license_tag_fee_flg string not null comment 'name: license tag fee flag
description: flag to indicate whether or not the item represents a fee for a license tag.
y = yes, n = no
source: petware pmri
dwpetnet.inventory.islicenseitem', 
item_prev_care_flg string not null comment 'name: item preventive care flag
description: a flag to indicate whether or not medical quality assurance considers this item as a preventive care item.  records with a value of "y" will be included in the refresh of the pet visit current preventive care group entity.
source: petware pmri
dwpetnet.inventory.inventoryrootid
dwpetnet.prevcareentityinventorylink.inventoryrootid
dwpetnet.prevcareentityinventorylink.defaultusefuldays', 
dea_controlled_flg string not null comment 'name: drug enforcement administration
description: a flag to indicate whether or not the item is controlled by the drug enforcement agency.
y = yes, n = no
source: petware pmri
dwpetnet.inventory.isdeacontrolled', 
item_male_flg string not null comment 'name: item male flag
description: a flag to indicate whether or not the item is suitable for male pets.
y = yes, n = no
source: petware pmri
dwpetnet.inventory.isformale', 
item_female_flg string not null comment 'name: item female flag
description: a flag to indicate whether or not the item is suitable for female pets.
y = yes, n = no
source: petware pmri
dwpetnet.inventory.isforfemale', 
item_neuter_flg string not null comment 'name: item neuter flag
description: a flag to indicate whether or not the item is suitable for neutered pets.
y = yes, n = no
source: petware pmri
dwpetnet.inventory.isforneutered', 
item_spay_flg string not null comment 'name: item spay flag
description: a flag to indicate whether or not the item is suitable for spayed pets.
y = yes, n = no
source: petware pmri
dwpetnet.inventory.isforspayed', 
item_canine_flg string not null comment 'name: item canine flag
description: a flag to indicate whether or not the item is suitable for canines.
y = yes, n = no
source: petware pmri
dwpetnet.inventory.islicenseitem', 
item_feline_flg string not null comment 'name: item feline flag
description: a flag to indicate whether or not the item is suitable for male felines.
y = yes, n = no
source: petware pmri
dwpetnet.inventory.islicenseitem', 
item_primary_typ_cd string comment 'name: item primary type code
description: system short name for the primary inventory type.
1             	services
12            	non-service items
2             	diagnostics
3             	medications
6             	surgical procedures
8             	medical/health care products
9             	wellness plans

source: petware pmri
dwpetnet.inventory.inventorytypeid', 
item_secondary_typ_cd string comment 'name: item secondary type code
description: system short name for the secondary inventory type.
ex:
32	dental care product
35	heartworm medication
36	intestinal medication
37	medical supply

source: petware pmri
dwpetnet.inventory.inventorycategoryid', 
dw_deleted_ind bigint not null comment 'name: data warehouse deleted indicator
description: dw metadata. an indicator set to 1 to designate when a particular key has been removed from a source system of record.
a value of 0 indicates that the key is still present in the source system of record.
source: derived by dw load process, by comparing source keys against target keys.', 
dw_job_id bigint not null comment 'name: data warehouse job identifier
description: dw metadata. an identifier for the data warehouse load process that last affected the record.
source: data warehouse database sequence generator', 
dw_load_dt timestamp not null comment 'name: data warehouse load date
description: dw metadata. the date the data warehouse load process that last affected the record.
this is the starting date/time of the process that last inserted or updated the record.
source: data warehouse local system date/time (us-pacific time zone).', 
audit_category_cd string comment 'name: audit category code
description: item audit category code.
a      	reg. adjust
a      	adjustment
j      	wp adjust
j      	wp adjustment (obsolete)
m      	wp member
m      	wp membership fee
n      	none
n      	normal
p      	wp payment (monthlyear)
p      	wp payment
r      	refund request
t      	sales tax
source: petware pmri
dwpetnet.inventory.auditcategorycode', 
audit_category_desc string comment 'name: audit category description
description: item audit category description.
a      	reg. adjust
a      	adjustment
j      	wp adjust
j      	wp adjustment (obsolete)
m      	wp member
m      	wp membership fee
n      	none
n      	normal
p      	wp payment (monthlyear)
p      	wp payment
r      	refund request
t      	sales tax
source: petware pmri
dwpetnet.auditcategorycode.description
dwpetnet.auditcategorycode.auditcategorycode
dwpetnet.inventory.auditcategorycode', 
default_packing_price_lvl_01 bigint comment 'name: default packing price level 1
description: item package price (level 1 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_packing_price_lvl_02 bigint comment 'name: default packing price level 2
description: item package price (level 2 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_packing_price_lvl_03 bigint comment 'name: default packing price level 3
description: item package price (level 3 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_packing_price_lvl_04 bigint comment 'name: default packing price level 4
description: item package price (level 4 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_packing_price_lvl_05 bigint comment 'name: default packing price level 5
description: item package price (level 5 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_packing_price_lvl_06 bigint comment 'name: default packing price level 6
description: item package price (level 6 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_packing_price_lvl_07 bigint comment 'name: default packing price level 7
description: item package price (level 7 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
reference_lab_flg string not null comment 'name: reference lab flag
description: a flag to indicate whether or not the item is a reference lab diagnostics test that will be sent out to an external lab.
y = yes, n = no
source: petware pmri
dwpetnet.inventory.isreferencelabtest', 
default_packing_price_lvl_08 bigint comment 'name: default packing price level 8
description: item package price (level 8 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_packing_price_lvl_09 bigint comment 'name: default packing price level 9
description: item package price (level 9 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_packing_price_lvl_10 bigint comment 'name: default packing price level 10
description: item package price (level 10 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_10 bigint comment 'name: default unit price level 10
description: item unit price (level 10 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_11 bigint comment 'name: default packing price level 11
description: item package price (level 11 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_11 bigint comment 'name: default unit price level 11
description: item unit price (level 11 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_12 bigint comment 'name: default packing price level 12
description: item package price (level 12 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_12 bigint comment 'name: default unit price level 12
description: item unit price (level 12 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_13 bigint comment 'name: default packing price level 13
description: item package price (level 13 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_13 bigint comment 'name: default unit price level 13
description: item unit price (level 13 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_14 bigint comment 'name: default packing price level 14
description: item package price (level 14 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_14 bigint comment 'name: default unit price level 14
description: item unit price (level 14 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_15 bigint comment 'name: default packing price level 15
description: item package price (level 15 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_15 bigint comment 'name: default unit price level 15
description: item unit price (level 15 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_16 bigint comment 'name: default packing price level 16
description: item package price (level 16 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_16 bigint comment 'name: default unit price level 16
description: item unit price (level 16 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_17 bigint comment 'name: default packing price level 17
description: item package price (level 17 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_17 bigint comment 'name: default unit price level 17
description: item unit price (level 17 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_18 bigint comment 'name: default packing price level 18
description: item package price (level 18 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_18 bigint comment 'name: default unit price level 18
description: item unit price (level 18 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_19 bigint comment 'name: default packing price level 19
description: item package price (level 19 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_19 bigint comment 'name: default unit price level 19
description: item unit price (level 19 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_20 bigint comment 'name: default packing price level 20
description: item package price (level 20 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_20 bigint comment 'name: default unit price level 20
description: item unit price (level 20 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
prdct_cost_amt bigint comment 'name: product cost amount
description: the product cost amount of the item.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventorycost.inventoryid
dwpetnet.inventorycost.productcost', 
default_packing_price_lvl_21 bigint comment 'name: default packing price level 21
description: item package price (level 21 us) - home delivery for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_21 bigint comment 'name: default unit price level 21
description: item unit price (level 21 us) - home delivery for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_22 bigint comment 'name: default packing price level 22
description: item package price (level 22 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_22 bigint comment 'name: default unit price level 22
description: item unit price (level 22 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_23 bigint comment 'name: default packing price level 23
description: item package price (level 23 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_23 bigint comment 'name: default unit price level 23
description: item unit price (level 23 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_24 bigint comment 'name: default packing price level 24
description: item package price (level 24 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_24 bigint comment 'name: default unit price level 24
description: item unit price (level 24 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_25 bigint comment 'name: default packing price level 25
description: item package price (level 25 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_25 bigint comment 'name: default unit price level 25
description: item unit price (level 25 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 

item_gl_account_num string comment 'name: item general ledger account number
description: the gl account number associated with the item
source: petware pmri
dwmisc.mmi_hr_gl_account.gl_account_num
dwmisc.mmi_hr_gl_account.pmri_inventory_category
dwpetnet.inventorycategory.description
dwpetnet.inventorycategory.inventorycategoryid
dwpetnet.inventory.inventorycategoryid', 
item_gl_account_descr string comment 'name: item general ledger account description
description: description of the gl account associated with the item
source: petware pmri
dwmisc.mmi_hr_gl_account.gl_account_descr
dwmisc.mmi_hr_gl_account.pmri_inventory_category
dwpetnet.inventorycategory.description
dwpetnet.inventorycategory.inventorycategoryid
dwpetnet.inventory.inventorycategoryid', 
item_updt_cost_with_price_flg string comment 'name: item update cost with price flag
description: indicates if the cost of goods field should be updated with the user entered unitprice value or if it should remain at the pmri set price
y = yes, n = no
source: petware pmri
dwpetnet.inventory.isupdatecostwithprice', 
item_user_editable_flg string comment 'name: item user editable flag
description: indicates if the users can control the unitprice assigned to a particular item via petware
y = yes, n = no
source: petware pmri
dwpetnet.inventory.isusereitable', 
non_sx_anesthesia_flg string comment 'name: non surgery anesthesia flag
description: indicates if anesthesia was used for a non-surgical procedure (e.g. y or n)
source: petware pmri
dwmisc.item_sx_anesthesia_excluded.non_sx_anesthesia_flg
dwmisc.item_sx_anesthesia_excluded.inventoryid
dwpetnet.inventory.inventoryid', 
healthy_sx_bndl_flg string comment 'name: healthy sex bundle flag
description: indicates if the procedure was done on a healthy pet; basically an elective procedure (e.g. y or n)
source: petware pmri
dwmisc.item_healthy_sx_bundle.healthy_sx_bndl_flg
dwmisc.item_healthy_sx_bundle.inventoryid
dwpetnet.inventory.inventoryid', 
item_tertiary_typ_cd bigint comment 'name: item tertiary type code
description: numeric value that corresponds to the item_tertiary_type (e.g. 2 is abdominal, 88 is dental prophylaxis)
ex:
44	antiviral
45	anxiolytic
46	appetite stimulant
48	behavior
49	benzodiazepine

source: petware pmri
dwpetnet.inventory.inventorytertiarycategoryid', 
item_tertiary_typ string comment 'name: item tertiary type
description: a grouping for medicine to use in reviewing items (e.g. abdominal, dental prophylaxis etc)
ex:
44	antiviral
45	anxiolytic
46	appetite stimulant
48	behavior
49	benzodiazepine

source: petware pmri
dwpetnet.inventory.inventorytertiarycategoryid
dwpetnet.inventorytertiarycategory.inventorytertiarycategoryid
dwpetnet.inventorytertiarycategory.description', 
management_category_cd bigint comment 'name: management category code
description: numeric value that corresponds to the management_category
ex:
1	anesthesia
2	behavior
3	cardiopulmonary
4	dental
5	dermatology
6	diagnostics
source: petware pmri
dwpetnet.managementcategory.managementcategoryid
dwpetnet.inventorymanagementcategorylink.managementcategoryid
dwpetnet.inventorymanagementcategorylink.inventoryrootid
dwpetnet.inventory.inventoryrootid', 
management_category string comment 'name: management category
description: a grouping for category management to use in reviewing items
ex:
1	anesthesia
2	behavior
3	cardiopulmonary
4	dental
5	dermatology
6	diagnostics
source: petware pmri
dwpetnet.managementcategory.description
dwpetnet.managementcategory.managementcategoryid
dwpetnet.inventorymanagementcategorylink.managementcategoryid
dwpetnet.inventorymanagementcategorylink.inventoryrootid
dwpetnet.inventory.inventoryrootid', 
default_mfg_nam string comment 'name: default manufacturer name
description: the default vendor name for same inventoryvendorlinkareaid as in inventory areaid, otherwise the default vendor name for global areaid else null
source: petware pmri
dwpetnet.vendor.name
dwpetnet.vendor.vendorid
dwpetnet.inventoryvendorlink.vendorid
dwpetnet.inventoryvendorlink.inventoryrootid
dwpetnet.inventoryvendorlink.areaid
dwpetnet.inventory.inventoryrootid
dwpetnet.inventory.areaid', 
documentation_cd bigint comment 'name: documentation code
description: numeric value corresponding to documentation_cd_desc which indicates if the item requires a label and/or a medical note (value, 1,2,3,4)
source: petware pmri
dwpetnet.inventory.documentationcode', 
documentation_cd_desc string comment 'name: documentation code description
description: indicates if the item requires a label and/or a medical note (values: nolabel_nomedicalnote, requiredlabel_requiredmedicalnote, optionallabel_requiredmedicalnote, and nolabel_requiredmedicalnote)
source: petware pmri
dwpetnet.codedescription.standarddescription
dwpetnet.codedescription.numericcode
dwpetnet.inventory.documentationcode', 
fda_prescription_flg string comment 'name: food and drug administration prescription flag
description: a flag (y or n) to indicate whether or not the item is an fda prescription
source: petware pmri
dwpetnet.inventory.isfdaprescription', 
item_formulary_nam string comment 'name: item formulary name
description: name of the method of delivery for medicines (e.g. bottle, capsule, injection, premeasured vial etc).
source: petware pmri
dwpetnet.unit.description
uses the following join conditions
dwpetnet.inventory.inventoryrootid = dwpetnet.inventorydrug.inventoryrootid
dwpetnet.inventorydrug.doseunitid = dwpetnet.unit.unitid', 
product_grp_nam string comment 'name: product group name
description: the group name of certain items (e.g. advantage, flea products, proheart etc). many values are null.
source: petware pmri
dwpetnet.rulecategory.description
uses the following join conditions
dwpetnet.inventory.inventoryrootid = dwpetnet.inventoryrulecategorylink.inventoryrootid
dwpetnet.inventoryrulecategorylink.rulecategoryid = dwpetnet.rulecategory.rulecategoryid', 
prev_care_useful_life_days bigint comment 'name: preventive care useful life days
description: the default value for the useful days of life for a given item.
source: petware pmri
dwpetnet.prevcareentityinventorylink.defaultusefuldays', 
item_unit string comment 'name: item unit
description: the description of the unit format for the item
ex:
bag
bottle
box
can
card
case

source: petware pmri
dwpetnet.inventory.note', 
item_src_system_cd string not null comment 'name: item source system code
description: an abbreviation to identify the source system where the record was found.  
valid values include:
   dw - data warehouse derived record;
   pn - petware (.net records);
   pw - petware (older foxpro records)', 
multiple_price_area_flg string comment 'name: multiple price area flag
description: a flag that is set to y if there are multiple price areas for a given inventoryid.
source: petware pmri
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventory.inventoryid', 
ft_hw_item_dose_cnt bigint comment 'name: flea tick heartworm item dose count
description: valid only for the 3 item secondary types: flea/tick medication, heartworm medication and flea/tick/heartworm.
source: petware pmri
trunc(max(dwpetnet.prevcareentityinventorylink.defaultusefuldays) / 30) ', 
delivery_method_nam string comment 'name: delivery method name
description: deliverymethod name. possible to get multiple delivery methods for given item.
source: petware pmri
dwpetnet.inventorydeliverymethodlink.deliverymethodid
dwpetnet.deliverymethod.deliverymethodid
dwpetnetdeliverymethod.description', 
dw_item_descriptiontext_id bigint comment 'name: data warehouse item description text identifier
description: textid that corresponds to a text used in petware.
source: petware pmri
inventory.descriptiontextid', 
item_descr_spanish_us string comment 'name: item description spanish united states
description: the textual us spanish description of the item.
source: petware pmri
inventory.descriptiontextid
dwpetnet.translatedtext.textid
dwpetnet.translatedtext.cultureid
cmd.tb_cmd_translated_text.spanish_us_text', 
item_descr_spanish_pr string comment 'name: item description spanish puerto rico
description: the textual puertorico spanish description of the item.
source: petware pmri
inventory.descriptiontextid
dwpetnet.translatedtext.textid
dwpetnet.translatedtext.cultureid
cmd.tb_cmd_translated_text.spanish_pr_text', 
item_areaid bigint comment 'name: item area identifier
description: areaid that belongs to the inventoryid.
source: petware pmri
dwpetnet.inventory.areaid', 
item_area_record_start_dt timestamp comment 'name: item area record start date
description: inventory area record start date.
source: petware pmri
dwpetnet.inventory.recordstartdate', 
item_area_record_end_dt timestamp comment 'name: item area record end date
description: inventory area record end date.
source: petware pmri
dwpetnet.inventory.reordenddate', 
item_start_dt timestamp comment 'name: item start date
description: inventoryid start date. when the item comes into effective
source: petware pmri
dwpetnet.inventory.startdate', 
item_end_dt timestamp comment 'name: item end date
description: inventoryid end date. when the item has expired and is no longer in use
source: petware pmri
dwpetnet.inventory.enddate', 
drug_form_desc string comment 'name: drug form description
description: the  "form" of the prescription drug in which the medication was provided in, be it topical, oral, etc. for diet it would be like cardiac.
source: petware pmri
dwpetnet.drugformdescription.description
uses the following join conditions
dwpetnet.inventory.inventoryrootid = dwpetnet.inventorydrug.inventoryrootid 
dwpetnet.inventorydrug.medicationid = dwpetnet.medication.medicationid
dwpetnet.medication.drugformid = dwpetnet.drugform.drugformid
dwpetnet.drugform.drugformdescriptionid  = dwpetnet.drugformdescription.drugformdescriptionid', 
default_packing_price_lvl_26 bigint comment 'name: default packing price level 26
description: item package price (level 26 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_26 bigint comment 'name: default unit price level 26
description: item unit price (level 26 us) - home delivery for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_27 bigint comment 'name: default packing price level 27
description: item package price (level 27 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_27 bigint comment 'name: default unit price level 27
description: item unit price (level 27 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_28 bigint comment 'name: default packing price level 28
description: item package price (level 28 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_28 bigint comment 'name: default unit price level 28
description: item unit price (level 28 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_29 bigint comment 'name: default packing price level 29
description: item package price (level 29 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_29 bigint comment 'name: default unit price level 29
description: item unit price (level 29 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_30 bigint comment 'name: default packing price level 30
description: item package price (level 30 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_30 bigint comment 'name: default unit price level 30
description: item unit price (level 30 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_31 bigint comment 'name: default packing price level 31
description: item package price (level 31 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_31 bigint comment 'name: default unit price level 31
description: item unit price (level 31 us) - home delivery for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_32 bigint comment 'name: default packing price level 32
description: item package price (level 32 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_32 bigint comment 'name: default unit price level 32
description: item unit price (level 32 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_33 bigint comment 'name: default packing price level 33
description: item package price (level 33 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_33 bigint comment 'name: default unit price level 33
description: item unit price (level 33 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_34 bigint comment 'name: default packing price level 34
description: item package price (level 34 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_34 bigint comment 'name: default unit price level 34
description: item unit price (level 34 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_35 bigint comment 'name: default packing price level 35
description: item package price (level 35 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_35 bigint comment 'name: default unit price level 35
description: item unit price (level 35 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_36 bigint comment 'name: default packing price level 36
description: item package price (level 36 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_36 bigint comment 'name: default unit price level 36
description: item unit price (level 36 us) - home delivery for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_37 bigint comment 'name: default packing price level 37
description: item package price (level 37 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_37 bigint comment 'name: default unit price level 37
description: item unit price (level 37 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_38 bigint comment 'name: default packing price level 38
description: item package price (level 38 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_38 bigint comment 'name: default unit price level 38
description: item unit price (level 38 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_39 bigint comment 'name: default packing price level 39
description: item package price (level 39 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_39 bigint comment 'name: default unit price level 39
description: item unit price (level 39 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_40 bigint comment 'name: default packing price level 40
description: item package price (level 40 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_40 bigint comment 'name: default unit price level 40
description: item unit price (level 40 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_41 bigint comment 'name: default packing price level 41
description: item package price (level 41 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_41 bigint comment 'name: default unit price level 41
description: item unit price (level 41 us) - home delivery for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_42 bigint comment 'name: default packing price level 42
description: item package price (level 42 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_42 bigint comment 'name: default unit price level 42
description: item unit price (level 42 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_43 bigint comment 'name: default packing price level 43
description: item package price (level 43 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_43 bigint comment 'name: default unit price level 43
description: item unit price (level 43 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_44 bigint comment 'name: default packing price level 44
description: item package price (level 44 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_44 bigint comment 'name: default unit price level 44
description: item unit price (level 44 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_45 bigint comment 'name: default packing price level 45
description: item package price (level 45 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_45 bigint comment 'name: default unit price level 45
description: item unit price (level 45 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_46 bigint comment 'name: default packing price level 46
description: item package price (level 46 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_46 bigint comment 'name: default unit price level 46
description: item unit price (level 46 us) - home delivery for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_47 bigint comment 'name: default packing price level 47
description: item package price (level 47 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_47 bigint comment 'name: default unit price level 47
description: item unit price (level 47 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_48 bigint comment 'name: default packing price level 48
description: item package price (level 48 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_48 bigint comment 'name: default unit price level 48
description: item unit price (level 48 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_49 bigint comment 'name: default packing price level 49
description: item package price (level 49 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_49 bigint comment 'name: default unit price level 49
description: item unit price (level 49 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_50 bigint comment 'name: default packing price level 50
description: item package price (level 50 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_50 bigint comment 'name: default unit price level 50
description: item unit price (level 50 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_51 bigint comment 'name: default packing price level 51
description: item package price (level 51 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_51 bigint comment 'name: default unit price level 51
description: item unit price (level 51 us) - home delivery for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_52 bigint comment 'name: default packing price level 52
description: item package price (level 52 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_52 bigint comment 'name: default unit price level 52
description: item unit price (level 52 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_53 bigint comment 'name: default packing price level 53
description: item package price (level 53 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_53 bigint comment 'name: default unit price level 53
description: item unit price (level 53 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_54 bigint comment 'name: default packing price level 54
description: item package price (level 54 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_54 bigint comment 'name: default unit price level 54
description: item unit price (level 54 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_55 bigint comment 'name: default packing price level 55
description: item package price (level 55 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_55 bigint comment 'name: default unit price level 55
description: item unit price (level 55 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_56 bigint comment 'name: default packing price level 56
description: item package price (level 56 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_56 bigint comment 'name: default unit price level 56
description: item unit price (level 56 us) - home delivery for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_57 bigint comment 'name: default packing price level 57
description: item package price (level 57 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_57 bigint comment 'name: default unit price level 57
description: item unit price (level 57 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_58 bigint comment 'name: default packing price level 58
description: item package price (level 58 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_58 bigint comment 'name: default unit price level 58
description: item unit price (level 58 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_59 bigint comment 'name: default packing price level 59
description: item package price (level 59 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_59 bigint comment 'name: default unit price level 59
description: item unit price (level 59 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_60 bigint comment 'name: default packing price level 60
description: item package price (level 60 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_60 bigint comment 'name: default unit price level 60
description: item unit price (level 60 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_61 bigint comment 'name: default packing price level 61
description: item package price (level 61 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_61 bigint comment 'name: default unit price level 61
description: item unit price (level 61 us) - home delivery for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_62 bigint comment 'name: default packing price level 62
description: item package price (level 62 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_62 bigint comment 'name: default unit price level 62
description: item unit price (level 62 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_63 bigint comment 'name: default packing price level 63
description: item package price (level 63 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_63 bigint comment 'name: default unit price level 63
description: item unit price (level 63 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_64 bigint comment 'name: default packing price level 64
description: item package price (level 64 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_64 bigint comment 'name: default unit price level 64
description: item unit price (level 64 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
default_packing_price_lvl_65 bigint comment 'name: default packing price level 65
description: item package price (level 65 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.packageprice', 
default_unit_price_lvl_65 bigint comment 'name: default unit price level 65
description: item unit price (level 65 us) for same inventorypriceareaid as inventory areaid.
source: petware pmri
dwpetnet.inventory.inventoryid
dwpetnet.inventory.areaid
dwpetnet.inventoryprice.inventoryid
dwpetnet.inventoryprice.areaid
dwpetnet.inventoryprice.pricelevelid
dwpetnet.inventoryprice.unitprice', 
item_shippable_flg string   comment  'name: item shippable flag 
description: a flag to indicate whether or not the item is shippable.
source:
dwpetnet.inventory.isshippable',

   comment on materialized view "cmn"."tb_cmd_current_item"  is 'name: inventory item dimension
description: the set of distinct items and services that may be referenced as a line item on an invoice.  this includes taxes, coupons and adjustments, as well as products and services.
grain: one row per inventory item, per historical change for that item
each inventory item will have multiple rows in this table. with a row for each change to any of its attributes.
each historical instance of a row will have an effective start and end date
the most recent version of a row is marked with a dw_curr_row_ind = 1
this table is loaded every day. note that currently, pmri is updated only every few weeks or so. in between releases, it is unlikely that any changes will occur in this table.
source: petware pmri
dwpetnet.inventory';

fw_createdts timestamp  comment 'the timestamp when the record was first loaded into the iron table',
fw_modifiedts timestamp  comment 'the timestamp when the record was modified and loaded in bronze table',
fw_filename string comment 'the name of the file from which the record was loaded',
rowhash string comment 'hash value generated from the contents of each row, used for detecting changes, deduplication, and maintaining data integrity in Delta Lake tables',
record_status string comment 'reflects the status of the bronze layer record, such as valid or invalid. we will push only valid records to silver layer',
constraint `cmn_tbcmdcurrentitem_pk` primary key (`inventoryid`) rely)
using delta
comment 'name: inventory item dimension
description: the set of distinct items and services that may be referenced as a line item on an invoice.  this includes taxes, coupons and adjustments, as well as products and services.
grain: one row per inventory item, per historical change for that item
each inventory item will have multiple rows in this table. with a row for each change to any of its attributes.
each historical instance of a row will have an effective start and end date
the most recent version of a row is marked with a dw_curr_row_ind = 1
this table is loaded every day. note that currently, pmri is updated only every few weeks or so. in between releases, it is unlikely that any changes will occur in this table.
source: petware pmri
dwpetnet.inventory'
tblproperties (
  'delta.checkpoint.writestatsasjson' = 'false',
  'delta.checkpoint.writestatsasstruct' = 'true',
  'delta.minreaderversion' = '1',
  'delta.minwriterversion' = '2',
  'delta.feature.allowColumnDefaults' = 'supported')

5. Create View for Gold Layer

In [0]:
%sql
create or replace view ${catlg.banfield_catalog}.bfdw_gold.cmn_tbcmdcurrentitem

as

select a.*
from ${catlg.banfield_catalog}.bfdw_silver.cmn_tbcmdcurrentitem a where 1 =1 

6. Create View for bfdw_date_quality for silver layer

In [0]:
%sql
create or replace view ${catlg.banfield_catalog}.bfdw_data_quality.cmn_tbcmdcurrentitem_silver_primary_key_exceptions

as

select a.*
from ${catlg.banfield_catalog}.bfdw_silver.cmn_tbcmdcurrentitem a where 1 !=1 


7. Create View for bfdw_data_quality for Bronze layers

In [0]:
%sql
create or replace view ${catlg.banfield_catalog}.bfdw_data_quality.cmn_tbcmdcurrentitem_bronze_record_status_exceptions

as
Select *  
from  ${catlg.banfield_catalog}.bfdw_bronze.cmn_tbcmdcurrentitem
where record_status != 'valid';